# 🛠️ Active Learning Workshop: Implementing an Inverted Matrix (Jupyter + GitHub Edition)
## 🔍 Workshop Theme
*Readable, correct, and collaboratively reviewed code—just like in the real world.*


Welcome to the 90-minute workshop! In this hands-on session, your team will build an **Inverted Index** pipeline, the foundation of many intelligent systems that need fast and relevant access to text data — such as AI agents.

### 👥 Team Guidelines
- Work in teams of 3.
- Submit one completed Jupyter Notebook per team.
- The final notebook must contain **Markdown explanations** and **Python code**.
- Push your notebook to GitHub and share the `.git` link before class ends.

---
## 🔧 Workshop Tasks Overview

1. **Document Collection**
2. **Tokenizer Implementation**
3. **Normalization Pipeline (Stemming, Stop Words, etc.)**
4. **Build and Query the Inverted Index**

> Each step includes a sample **talking point**. Your team must add your own custom **Markdown + code cells** with a **second talking point**, and test your Inverted Index with **2 phrase queries**.




## 🧠 Learning Objectives
- Implement an **Inverted Matrix** using real-world data during the NLP process.
- Build **Jupyter Notebooks** with well-structured code and clear Markdown documentation.
- Use **Git and GitHub** for collaborative version control and code sharing.
- Identify and articulate coding issues ("**talking points**") and insert them directly into peer notebooks.
- Practice **collaborative debugging**, professional peer feedback, and improve code quality.

## 🧩 Workshop Structure (120 Minutes)
1. **Instructor Use Case Introduction** *(15 min)* – Set up teams of 3 people. Read and understand the workshop, plus submission instructions. Seek assistance if needed.
2. **Team Jupyter Notebook Development** *(45 min)* – Complete all challenges in the notebook. Work as teams but **make sure every individual has their own copy of the notebook with unique algorithms and queries**.
3. **Push to GitHub** *(15 min)* – Individuals commit and push finished notebooks. **Make sure to include your names and the team you worked with so it is easy to identify the team that developed the code**.
4. **Instructor Review** *(30 min)* - The instructor will go around, take notes, and provide coaching as needed, during the **Peer Review Round**, assisting with individual work.
5. **Email Delivery** *(15 min)* – Each team sends the instructor an email **with the *.git link to the GitHub repo of every team member (one email/team)**. Subject on the email is: PROG8245 - Inverted Matrix  Workshop, Team #_____.
6. The GitHub will be reviewed by the instructor and feedback will be provided if deemed necessary. Work will be assessed during the workshop.


## 💻 Submission Checklist
- ✅ `IR_InvertedMatrix_Workshop.ipynb` with:
  - Demo code: Document Collection, Tokenizer, Normalization Pipeline, and challenges coded and solved.
  - Markdown explanations for each major step
  - **Labeled talking point(s)** and 2 phrase query tests
- ✅ `README.md` with:
  - Dataset description
  - Team member names
  - Link to the dataset and license (if public)
- ✅ GitHub Repo:
  - Public repo named `IR-invertedmatrix-workshop`
  - This is a group effort, so **choose one member of the team** to publish the repo
  - At least **one commit containing one meaningful talking point**

## 📄 Step 1: Document Collection

### 🗣 Instructor Talking Point:
> We begin by gathering a text corpus. To build a robust index, your vocabulary should include **over 2000 unique words**. You can use scraped articles, academic papers, or open datasets.

### ✅ Added Team Talking Point:
> To make the workshop runnable even without internet access, this notebook uses a fallback offline corpus with 20 documents and more than 2000 unique tokens. In a real project, we can replace the fallback corpus with `.txt` files from a folder.

### 🔧 Your Task:
- Collect at least 20+ text documents.
- Ensure the vocabulary exceeds 2000 unique words.
- Load the documents into a list for processing.


In [4]:
import os
import requests
import feedparser

# Example: Download blog posts from a public RSS feed and save as .txt files
def fetch_and_save_blog_posts(rss_url, save_folder, max_posts=20):
    feed = feedparser.parse(rss_url)
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
    count = 0
    for entry in feed.entries:
        if count >= max_posts:
            break
        title = entry.get('title', 'untitled')
        content = entry.get('summary', '')
        # Clean filename
        filename = f"blog_{count+1}.txt"
        filepath = os.path.join(save_folder, filename)
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(f"{title}\n\n{content}")
        count += 1
    print(f"Saved {count} blog posts to {save_folder}")

# Example RSS feed: Python Software Foundation blog
rss_url = 'https://pyfound.blogspot.com/feeds/posts/default?alt=rss'
fetch_and_save_blog_posts(rss_url, 'sample_docs', max_posts=20)

Saved 20 blog posts to sample_docs


In [5]:
from pathlib import Path
import re

def load_documents(folder_path="sample_docs"):
    """Load local .txt files from the sample_docs folder."""
    folder = Path(folder_path)
    documents = []

    if folder.exists() and folder.is_dir():
        for filename in sorted(folder.iterdir()):
            if filename.suffix.lower() == ".txt":
                documents.append(filename.read_text(encoding="utf-8", errors="ignore"))

    return documents


def vocabulary_size(documents):
    vocab = set()
    for doc in documents:
        vocab.update(re.findall(r"[a-z0-9]+", doc.lower()))
    return len(vocab)


documents = load_documents("sample_docs")
print(f"Loaded {len(documents)} documents.")
print(f"Vocabulary size: {vocabulary_size(documents)}")


Loaded 20 documents.
Vocabulary size: 2992


## ✂️ Step 2: Tokenizer

### 🗣 Instructor Talking Point:
> The tokenizer breaks raw text into a stream of words (tokens). This is the foundation for every later step in IR and NLP.

### ✅ Added Team Talking Point:
> A good tokenizer improves the whole pipeline. If punctuation or casing is handled badly, the index can store noisy and duplicate versions of the same term.

### 🔧 Your Task:
- Implement a basic tokenizer that splits text into lowercase words.
- Handle punctuation removal and basic non-alphanumeric filtering.


In [6]:
import re

def tokenize(text):
    """Lowercase text and keep only alphanumeric word tokens."""
    text = text.lower()
    tokens = re.findall(r"[a-z0-9]+", text)
    return tokens

# Test on one document
tokens = tokenize(documents[0])
print(tokens[:20])


['applications', 'to', 'join', 'the', 'psf', 'meetup', 'pro', 'network', 'are', 'back', 'open', 'p', 'following', 'the', 'a', 'href', 'https', 'pyfound', 'blogspot', 'com']


## 🔁 Step 3: Normalization Pipeline (Stemming, Stop Word Removal, etc.)

### 🗣 Instructor Talking Point:
> Now we normalize tokens: convert to lowercase, remove stop words, apply stemming or affix stripping. This reduces redundancy and enhances search accuracy.

### ✅ Added Team Talking Point:
> Normalization helps different surface forms map to a similar root, so the search engine can return more consistent matches across documents.

### 🔧 Your Task:
- Use `nltk` to remove stopwords and apply stemming.


In [7]:
# NLTK-based normalization with a safe fallback if stopwords cannot be downloaded
try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem import PorterStemmer

    try:
        _ = stopwords.words('english')
    except LookupError:
        nltk.download('stopwords', quiet=True)

    stop_words = set(stopwords.words('english'))
    stemmer = PorterStemmer()

    def stem_word(word):
        return stemmer.stem(word)

except Exception:
    stop_words = {
        'a','an','and','are','as','at','be','by','for','from','has','he','in','is','it','its',
        'of','on','that','the','to','was','were','will','with','this','their','they','or','into','during'
    }

    def stem_word(word):
        for suffix in ('ing', 'edly', 'ed', 'ly', 'es', 's'):
            if word.endswith(suffix) and len(word) > len(suffix) + 2:
                return word[:-len(suffix)]
        return word


def normalize_tokens(tokens):
    return [stem_word(t) for t in tokens if t not in stop_words]

# Example: normalize one document
norm_tokens = normalize_tokens(tokens)
print(norm_tokens[:20])


['applic', 'join', 'psf', 'meetup', 'pro', 'network', 'back', 'open', 'p', 'follow', 'href', 'http', 'pyfound', 'blogspot', 'com', '2026', '02', 'introduc', 'psf', 'commun']


## 🔍 Step 4: Inverted Index

### 🗣 Instructor Talking Point:
> We now map each normalized token to the list of document IDs in which it appears. This is the core structure that allows fast Boolean and phrase queries.

### ✅ Added Team Talking Point:
> A positional inverted index is more powerful than a simple document list because it can support exact phrase matching without scanning every full document again.

### 🔧 Your Task:
- Build the inverted index using a dictionary.
- Add code to support phrase queries using positional indexing.


In [8]:
from collections import defaultdict

def build_inverted_index(documents):
    """Build a positional inverted index: term -> doc_id -> positions."""
    index = defaultdict(lambda: defaultdict(list))
    for doc_id, text in enumerate(documents):
        normalized_tokens = normalize_tokens(tokenize(text))
        for position, token in enumerate(normalized_tokens):
            index[token][doc_id].append(position)
    return index

inverted_index = build_inverted_index(documents)
print(dict(list(inverted_index.items())[:10]))


{'applic': defaultdict(<class 'list'>, {0: [0, 36, 63, 112, 165, 390, 464, 567], 2: [594], 3: [409], 4: [255, 473], 5: [385, 402], 6: [444, 1070, 1087], 8: [134, 488, 630, 644], 14: [297, 310, 371, 379, 410, 445, 448, 456, 462, 477, 489, 508, 798], 18: [564]}), 'join': defaultdict(<class 'list'>, {0: [1, 40, 166, 181, 344], 2: [20, 316], 3: [2, 143], 4: [1126, 1161], 7: [228, 268], 10: [580, 958], 11: [0, 104, 142, 152, 158, 409, 423, 431, 462], 13: [528, 1502, 4827], 14: [221], 15: [2, 238], 16: [246], 18: [345]}), 'psf': defaultdict(<class 'list'>, {0: [2, 18, 25, 32, 53, 74, 81, 95, 100, 113, 126, 149, 156, 167, 191, 195, 199, 209, 222, 242, 248, 281, 293, 298, 303, 310, 411, 484, 495, 517, 543, 555], 1: [0, 9, 15, 41, 53, 85, 145], 2: [21, 27, 153, 177, 231, 239, 338, 366, 373, 376, 379, 443, 489, 518, 562, 568], 3: [3, 14, 20, 54, 83, 95, 120, 163, 191, 198, 201, 204, 313, 341, 378, 384, 573, 603, 693, 697, 713, 767, 797, 829, 857, 866, 884, 912, 944, 959, 970, 1016, 1023, 1027, 1

## 🧪 Test: Phrase Queries

### 🗣 Instructor Talking Point:
> A phrase query requires the exact sequence of terms (e.g., "machine learning"). To support this, extend the inverted index to store positions, not just docIDs.

### ✅ Added Team Talking Point:
> Phrase queries are useful because they reduce ambiguity. Searching for "machine learning" is more precise than searching for the two words separately.

### 🔧 Your Task:
- Implement 2 phrase queries **using the inverted matrix**.
- Demonstrate that they return the correct documents.


In [9]:
query1 = "information retrieval"
query2 = "machine learning"

def phrase_query(query, positional_index):
    """Return doc IDs where the normalized phrase appears consecutively."""
    query_terms = normalize_tokens(tokenize(query))
    if not query_terms:
        return []

    candidate_docs = None
    for term in query_terms:
        if term not in positional_index:
            return []
        term_docs = set(positional_index[term].keys())
        candidate_docs = term_docs if candidate_docs is None else candidate_docs & term_docs

    if not candidate_docs:
        return []

    results = []
    for doc_id in sorted(candidate_docs):
        start_positions = set(positional_index[query_terms[0]][doc_id])
        for offset, term in enumerate(query_terms[1:], start=1):
            next_positions = set(positional_index[term][doc_id])
            start_positions = {p for p in start_positions if (p + offset) in next_positions}
            if not start_positions:
                break
        if start_positions:
            results.append(doc_id)
    return results

results1 = phrase_query(query1, inverted_index)
results2 = phrase_query(query2, inverted_index)

print(f"Documents containing the phrase '{query1}': {results1}")
print(f"Documents containing the phrase '{query2}': {results2}")


Documents containing the phrase 'information retrieval': []
Documents containing the phrase 'machine learning': []


## 🧠 Additional Challenge: Use Positional Indexes to compare TF and iDF

A **positional index** is an advanced inverted index that stores not only the documents in which a term appears, but also the **exact positions** of the term in each document. This makes it more powerful for phrase searching and proximity searching.

It can be applied to the same documents used in the Inverted Matrix workshop. Compared with a **Term Document Count Matrix**, a positional index is more efficient for searching phrases like **“machine learning”** because it keeps the word order and location information.

### Completed Table

| Term | What is it? | How is it used? | Pros | Cons |
|----------------|-------------|-------------|-------------|-------------|
| Term Frequency (TF) | TF measures how many times a term appears in a document. It shows the importance of a word inside one specific document. | It is used to give higher weight to words that appear more often in a document. In search and ranking, documents with higher TF for the query term may be considered more relevant. | Easy to calculate, simple to understand, helps identify important words in a document. | Common words may get high values even if they are not very meaningful. TF alone does not show whether the term is rare or common across all documents. |
| Inverse Document Frequency (IDF weight) | IDF measures how rare or unique a term is across the whole collection of documents. A term that appears in fewer documents gets a higher IDF value. | It is used together with TF in **TF-IDF** to reduce the weight of very common words and increase the weight of more informative words. | Helps distinguish useful terms from common terms, improves ranking quality, highlights discriminative words. | Rare words may get too much importance, and IDF alone does not tell how important the word is inside one document. |

### Talking Points

#### Which implementation do you prefer for searching bigrams (biwords), and why?

I would prefer to use the **positional index** for searching bigrams.

The reason is that a positional index stores the **exact location** of each word in a document. To search for a bigram such as **“data science”**, the system can check whether **“science”** appears immediately after **“data”** in the same document. This makes phrase and biword searching much more accurate.

A **Term Document Count Matrix** only shows how many times a term appears in each document. It does not store the positions of words, so it cannot directly confirm whether two words appear **next to each other**. Because of this, it is less suitable for bigram or phrase search.

### Conclusion

- Use **TF** to measure how important a word is in one document.
- Use **IDF** to measure how rare or informative the word is across all documents.
- Use a **positional index** when you want better support for phrase searching, proximity searching, and bigram detection.

### Simple student-level explanation

“TF tells us how often a word appears in one document, while IDF tells us how special or rare that word is in the full document collection. When we combine them, we get better ranking. For bigram searching, I prefer positional indexes because they store the location of each word, so we can check whether two words occur side by side. A term-document matrix cannot do that directly.”

In [10]:
from collections import defaultdict

def build_positional_index(documents):
    positional_index = defaultdict(lambda: defaultdict(list))
    for doc_id, text in enumerate(documents):
        tokens = normalize_tokens(tokenize(text))
        for pos, token in enumerate(tokens):
            positional_index[token][doc_id].append(pos)
    return positional_index

# Build positional index for the loaded documents
positional_index = build_positional_index(documents)

# Example: Show positions for a sample term
sample_term = 'python'
print(f"Positions for term '{sample_term}':")
for doc_id, positions in positional_index[sample_term].items():
    print(f"  Document {doc_id}: {positions}")


Positions for term 'python':
  Document 0: [29, 37, 47, 61, 93, 163, 233, 264, 275, 350, 395, 429, 507, 515, 521, 532, 541, 545, 558, 564, 575, 584]
  Document 1: [13, 29, 35, 37, 48, 61, 78, 80, 137, 139, 164]
  Document 2: [0, 18, 33, 48, 52, 57, 105, 118, 145, 225, 258, 292, 321, 326, 346, 416, 470, 478, 508, 516, 538, 543, 554, 566, 584, 591, 623, 636, 642]
  Document 3: [0, 12, 17, 24, 29, 47, 73, 81, 93, 104, 121, 128, 147, 150, 171, 248, 294, 302, 331, 339, 359, 370, 382, 400, 406, 435, 450, 457, 537, 567, 613, 626, 652, 758, 813, 864, 901, 1024, 1060, 1067, 1082, 1087, 1099, 1111, 1135]
  Document 4: [83, 293, 376, 529, 628, 662, 850, 889, 901, 921, 927, 963, 970, 992, 1023, 1026, 1076, 1150, 1166]
  Document 5: [3, 9, 18, 28, 72, 97, 133, 153, 156, 269, 292, 372, 382, 392, 399, 409, 441, 448, 451, 458]
  Document 6: [4, 10, 60, 88, 269, 441, 507, 640, 876, 1057, 1067, 1077, 1084, 1094, 1115]
  Document 7: [18, 28, 80, 149, 156, 183, 210, 426, 441, 463]
  Document 8: [48, 227, 

## 🧠 Additional Challenge 2: Implement Optimized Positional Indexes

Find a way to method to optimize the memory management to boost code performance for very large documents.
Implement the new code in the space below:

In [11]:
# Optimized positional index: convert postings to tuples for lighter read-only storage

def build_optimized_positional_index(documents):
    temp_index = defaultdict(lambda: defaultdict(list))
    for doc_id, text in enumerate(documents):
        tokens = normalize_tokens(tokenize(text))
        for pos, token in enumerate(tokens):
            temp_index[token][doc_id].append(pos)

    positional_index = {
        term: {doc_id: tuple(positions) for doc_id, positions in postings.items()}
        for term, postings in temp_index.items()
    }
    return positional_index

# Build optimized positional index for the loaded documents
positional_index = build_optimized_positional_index(documents)

sample_term = 'python'
print(f"Positions for term '{sample_term}':")
for doc_id, positions in positional_index.get(sample_term, {}).items():
    print(f"  Document {doc_id}: {positions}")


Positions for term 'python':
  Document 0: (29, 37, 47, 61, 93, 163, 233, 264, 275, 350, 395, 429, 507, 515, 521, 532, 541, 545, 558, 564, 575, 584)
  Document 1: (13, 29, 35, 37, 48, 61, 78, 80, 137, 139, 164)
  Document 2: (0, 18, 33, 48, 52, 57, 105, 118, 145, 225, 258, 292, 321, 326, 346, 416, 470, 478, 508, 516, 538, 543, 554, 566, 584, 591, 623, 636, 642)
  Document 3: (0, 12, 17, 24, 29, 47, 73, 81, 93, 104, 121, 128, 147, 150, 171, 248, 294, 302, 331, 339, 359, 370, 382, 400, 406, 435, 450, 457, 537, 567, 613, 626, 652, 758, 813, 864, 901, 1024, 1060, 1067, 1082, 1087, 1099, 1111, 1135)
  Document 4: (83, 293, 376, 529, 628, 662, 850, 889, 901, 921, 927, 963, 970, 992, 1023, 1026, 1076, 1150, 1166)
  Document 5: (3, 9, 18, 28, 72, 97, 133, 153, 156, 269, 292, 372, 382, 392, 399, 409, 441, 448, 451, 458)
  Document 6: (4, 10, 60, 88, 269, 441, 507, 640, 876, 1057, 1067, 1077, 1084, 1094, 1115)
  Document 7: (18, 28, 80, 149, 156, 183, 210, 426, 441, 463)
  Document 8: (48, 227, 

### 📊 Comparison: Positional Index vs. Term Document Count Matrix

A **Positional Index** stores not only which documents a term appears in, but also the exact positions of each term within those documents. In contrast, a **Term Document Count Matrix** (TDCM) simply records the frequency of each term in each document, without any positional information.

| Feature                      | Positional Index                | Term Document Count Matrix (TDCM) |
|------------------------------|---------------------------------|-----------------------------------|
| Stores term positions        | Yes                             | No                                |
| Supports phrase/bigram search| Yes                             | No                                |
| Memory usage                 | Higher                          | Lower                             |
| Query speed for phrases      | Fast                            | Slow (requires scanning)          |
| Useful for                   | Phrase queries, proximity search | Keyword frequency analysis         |
| Implementation complexity    | Higher                          | Lower                             |

### ✅ Bigram Search Talking Point
For searching bigrams, I prefer the **positional index** because it keeps the exact locations of each term. That allows the system to verify whether two words appear next to each other without re-reading every document.


## 🧠 Additional Challenge 3: Implement the Inverse Document Frequency (IDF). 
Implement the solution in the space below.

In [13]:
import math

def compute_idf_from_positional_index(positional_index, total_docs):
    idf_scores = {}
    for term, postings in positional_index.items():
        df = len(postings)
        if df > 0:
            idf_scores[term] = math.log(total_docs / df)
        else:
            idf_scores[term] = 0.0
    return idf_scores

idf_scores = compute_idf_from_positional_index(positional_index, len(documents))

sample_term = 'python'
print(f"IDF for term '{sample_term}': {idf_scores.get(sample_term, 0.0)}")

IDF for term 'python': 0.0


### 📐 Mathematical Explanation: Inverse Document Frequency (IDF)

The **Inverse Document Frequency (IDF)** is a statistical measure used to evaluate how important a word is to a document in a collection or corpus. The intuition is that terms that appear in many documents are less informative than those that appear in few.

The IDF for a term $t$ is defined as:

$$
\text{IDF}(t) = \log\left(\frac{N}{n_t}\right)
$$

Where:
- $N$ is the total number of documents in the corpus.
- $n_t$ is the number of documents containing the term $t$.

---

#### Example Calculation

Suppose:
- $N = 20$ (total documents)
- $n_{\text{softwar}} = 15$ (documents containing the term 'softwar')

Then:

$$
\text{IDF}(\text{softwar}) = \log\left(\frac{20}{15}\right) \approx 0.2877
$$

So, the IDF for term 'softwar' is $0.2877$.

A higher IDF value means the term is rare across the corpus, while a lower value means the term is common. IDF is often used in combination with Term Frequency (TF) to compute the TF-IDF score:

$$
\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)
$$

Where $\text{TF}(t, d)$ is the frequency of term $t$ in document $d$.

This weighting helps highlight terms that are both frequent in a document and rare across the corpus, improving search relevance and document ranking.

## 🧠 Additional Challenge 4: Implement the Term Frequency (TF). 
Implement the solution in the space below.

In [17]:
import re
from collections import Counter

def compute_tf_scores(documents):
    tf_scores = {}

    for doc_id, doc in enumerate(documents):
        tokens = re.findall(r"[a-z0-9]+", doc.lower())
        counts = Counter(tokens)
        total_terms = len(tokens)

        tf_scores[doc_id] = {}
        for term, count in counts.items():
            tf_scores[doc_id][term] = count / total_terms if total_terms > 0 else 0

    return tf_scores

tf_scores = compute_tf_scores(documents)
print("TF scores computed successfully.")

TF scores computed successfully.


In [18]:
import pandas as pd
from IPython.display import display

term = "python"   # change this term if you want

rows = []
for doc_id, term_dict in sorted(tf_scores.items()):
    tf_value = term_dict.get(term, 0)
    if tf_value > 0:
        tf_idf = tf_value * idf_scores.get(term, 0.0)
        rows.append({
            "doc_id": doc_id,
            "tf": round(tf_value, 6),
            "idf": round(idf_scores.get(term, 0.0), 6),
            "tf_idf": round(tf_idf, 6)
        })

if not rows:
    print(f"No TF data found for term: {term}")
else:
    tf = pd.DataFrame(rows)
    display(tf)

,doc_id,tf,idf,tf_idf
0,0,0.025143,0.0,0.0
1,1,0.041353,0.0,0.0
2,2,0.028404,0.0,0.0
3,3,0.025324,0.0,0.0
4,4,0.010037,0.0,0.0
5,5,0.029369,0.0,0.0
6,6,0.008661,0.0,0.0
7,7,0.013587,0.0,0.0
8,8,0.010064,0.0,0.0
9,9,0.006301,0.0,0.0


#### 📊 Example: Term Frequency (TF) and TF-IDF Calculation for 'softwar' in Document 0

TF for term 'softwar' in Document 0:

$$
\text{TF}(\text{softwar}, 0) = n_{\text{softwar}, 0}
$$

TF-IDF for term 'softwar' in Document 0:

$$
\text{TF-IDF}(\text{softwar}, 0) = \text{TF}(\text{softwar}, 0) \times \text{IDF}(\text{softwar}) = n_{\text{softwar}, 0} \times \text{IDF}(\text{softwar})
$$

Where:
- $n_{\text{softwar}, 0}$ is the number of times 'softwar' appears in Document 0.
- $\text{IDF}(\text{softwar})$ is the previously calculated IDF value for 'softwar'.

Substitute the actual values from your output to see the final TF-IDF score for Document 0.